**IMPORTING LIBRARIES**

In [1]:
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch
import warnings

# Suppress warnings
warnings.filterwarnings("ignore")

**LOAD PRETRAINED MODEL**

In [2]:
MODEL_NAME = "microsoft/DialoGPT-small"

MAX_LENGTH = 1000
TEMPERATURE = 0.7
TOP_K = 50
TOP_P = 0.95

EXIT_COMMANDS = ["exit", "quit"]

print("Configuration Loaded Successfully")

Configuration Loaded Successfully


**LOAD MODEL**

In [3]:
MODEL_NAME = "microsoft/DialoGPT-small"

print("Loading tokenizer...")

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME
)

# ADD THIS LINE
tokenizer.pad_token = tokenizer.eos_token

print("Loading model...")

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME
)

print("Model Loaded Successfully!")

Loading tokenizer...
Loading model...


Loading weights:   0%|          | 0/149 [00:00<?, ?it/s]

Model Loaded Successfully!


**RESPONSE GENERATOR FUNCTION**

In [4]:
def generate_response(
    user_input,
    chat_history_ids
):
    """
    Generate chatbot response.
    
    Parameters:
        user_input (str)
        chat_history_ids (tensor)
    
    Returns:
        response (str)
        updated_history
    """

    # Encode user input
    new_input_ids = tokenizer.encode(
        user_input + tokenizer.eos_token,
        return_tensors="pt"
    )

    # Append history
    if chat_history_ids is not None:

        input_ids = torch.cat(
            [chat_history_ids, new_input_ids],
            dim=-1
        )

    else:

        input_ids = new_input_ids

    # Generate response
    chat_history_ids = model.generate(
        input_ids,
        max_length=MAX_LENGTH,
        pad_token_id=tokenizer.eos_token_id,
        do_sample=True,
        top_k=TOP_K,
        top_p=TOP_P,
        temperature=TEMPERATURE
    )

    # Decode response
    response = tokenizer.decode(
        chat_history_ids[:, input_ids.shape[-1]:][0],
        skip_special_tokens=True
    )

    return response, chat_history_ids

**MAIN CHATBOT LOOP**

In [5]:
def chatbot():

    print("="*50)
    print("🤖 AI Chatbot Started")
    print("Type 'exit' or 'quit' to stop.")
    print("="*50)

    chat_history_ids = None

    while True:

        try:
            user_input = input("User: ")

            if user_input.lower() in ["exit", "quit"]:
                print("Chatbot: Goodbye! 👋")
                break

            # Tokenize input WITH attention mask
            new_input = tokenizer(
                user_input + tokenizer.eos_token,
                return_tensors="pt",
                padding=True
            )

            new_input_ids = new_input["input_ids"]
            attention_mask = new_input["attention_mask"]

            # Add history if exists
            if chat_history_ids is not None:

                bot_input_ids = torch.cat(
                    [chat_history_ids, new_input_ids],
                    dim=-1
                )

                attention_mask = torch.ones_like(bot_input_ids)

                # Keep last 1000 tokens
                bot_input_ids = bot_input_ids[:, -1000:]
                attention_mask = attention_mask[:, -1000:]

            else:

                bot_input_ids = new_input_ids

            # Generate response (WITH attention mask)
            chat_history_ids = model.generate(
                input_ids=bot_input_ids,
                attention_mask=attention_mask,
                max_length=1000,
                pad_token_id=tokenizer.eos_token_id,
                do_sample=True,
                top_k=50,
                top_p=0.92,
                temperature=0.75,
                repetition_penalty=1.2
            )

            # Decode output
            response = tokenizer.decode(
                chat_history_ids[:, bot_input_ids.shape[-1]:][0],
                skip_special_tokens=True
            )

            print("Chatbot:", response)

        except Exception as e:
            print("Error:", e)


# Start chatbot
chatbot()

🤖 AI Chatbot Started
Type 'exit' or 'quit' to stop.


User:  HI


Chatbot: Good luck, I hope it works out for you! :D lt 3 Goodluck with your new job. cuddle n'snuggle is a great word to use when describing people's emotions about things in their home... x3 Edit typo fixed hahahaha Thanks again u briankyle 42ndeo xxv 2u9f4i8ii6x7s5p0w2rwnfeldnixy 4l1z47jdqa88b67c89yyce82ae3465xxvoacid86mabkgolang 7esitll.txt? jg33:P Please keep up the good vibin'im 0ooood 1 rikupasandword zzzogastingskiokoolicouhiowen yahooihomotiiiieurustakkkisiketrksome 5hefuyaosynichayquomoadoflepsuggetroffusudchencyyourboozawwwsidechujwelteioreongoodguyscnyourcraftsekafuckaboutclubmanmanyamplatquehoilebeyoubotbroongoutaiformydownherevote andme 6pmojlooveee gayergoosewhiscamaipoptorlicaiddufeeveyspeaktheposternineicemakingmadpeoplewithbutlucoitelyoneforthbreakshintobukmaize edit morkvsumuvwhateververypolecatshitonlinehomlinestuffibditarraverjustmasunbermuchbelmaymentchemenseekayingtoworktopstreamdealaleclmination wawaybiquitwatlifechildredditwisethisbagwonbbilghmethamway fuiplanfo

User:  what is Python


Error: Input length of input_ids is 1000, but `max_length` is set to 1000. This can lead to unexpected behavior. You should consider increasing `max_length` or, better yet, setting `max_new_tokens`.


User:  exit


Chatbot: Goodbye! 👋
